# NTL Word-choice C0/C1/C2 — official-only, checkpointable CPU/Colab pilot
This is a **mechanism falsification**, not a champion score estimate. No external synthetic model outputs. No Kaggle submission. Free Colab interactive execution only; no unattended remote worker. It uses exactly the same frozen prompt folds, training rows, labels, HashingVectorizer dimension, SGD loss/alpha/epochs and random seeds across three arms. C0 unigrams, C1 unigrams plus adjacent context, C2 unigrams plus within-passage shuffled adjacent context. This is a *linear pilot*, **not** the previously proposed RoBERTa/LoRA reader. Do not claim model capacity equality in effective features; feature dimension and optimizer are held constant, while feature distributions differ.

In [ ]:
from pathlib import Path
import os, json, hashlib, random, collections, importlib.util
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import HashingVectorizer
from sklearn.linear_model import SGDClassifier
from sklearn.metrics import balanced_accuracy_score
import joblib
# Run interactively in Colab after cloning the repo (public source only):
# !git clone --branch research/attribution-bottleneck-review-20261106 https://github.com/kolihhan/name-kaggle.git
REPO=Path('/content/name-kaggle') if Path('/content/name-kaggle').exists() else Path('.')
DATA=Path('/content/ntl_private/train.csv') # manually mount/upload OFFICIAL train.csv; never commit it
FOLDS=Path('/content/ntl_private/frozen_prompt_folds.json') # exact existing permanent prompt->fold map
OUT=Path('/content/ntl_private/wordchoice_pilot')
OUT.mkdir(parents=True,exist_ok=True)
SEED=20261121; N_FEATURES=2**18; EPOCHS=3; ALPHA=1e-5
spec=importlib.util.spec_from_file_location('ntl_gate',REPO/'scripts/ntl_context_wordchoice_gate_20261121.py')
gate_mod=importlib.util.module_from_spec(spec); spec.loader.exec_module(gate_mod)
print('CPU/GPU:',os.environ.get('COLAB_GPU','CPU'), 'DATA present:',DATA.exists(), 'FOLDS present:',FOLDS.exists())


In [ ]:
def normalize_words(text):
    return str(text).split()

def context_features(text, arm, row_id):
    words=normalize_words(text)
    if not words: return 'EMPTY'
    # A deterministic token permutation destroys adjacency but keeps each document's unigram inventory.
    if arm=='C2':
        rng=random.Random(int.from_bytes(hashlib.sha256((str(SEED)+'|'+str(row_id)).encode()).digest()[:8],'big'))
        words=words.copy(); rng.shuffle(words)
    # C0 is strictly bag-of-words; C1 and C2 use the same capacity / feature templates.
    uni=['u:'+w for w in normalize_words(text)]
    if arm=='C0': return ' '.join(uni)
    return ' '.join(uni+['b:'+a+'~'+b for a,b in zip(words,words[1:])])

def fit_one(train, valid, arm, fold):
    vectorizer=HashingVectorizer(n_features=N_FEATURES,alternate_sign=False,norm='l2',token_pattern=r'(?u)\\b\\w+\\b')
    # Fixed class set and identical optimization. No use of held-out labels during fitting.
    xtrain=vectorizer.transform(context_features(t,arm,i) for t,i in zip(train.text,train.id))
    xvalid=vectorizer.transform(context_features(t,arm,i) for t,i in zip(valid.text,valid.id))
    model=SGDClassifier(loss='log_loss',alpha=ALPHA,max_iter=EPOCHS,tol=None,random_state=SEED,shuffle=True)
    model.fit(xtrain,train.label.astype(int))
    pred=model.predict(xvalid).astype(int)
    checkpoint={'model':model,'fold':int(fold),'arm':arm,'train_ids':train.id.astype(str).tolist(),
                'train_prompt_ids':sorted(set(train.prompt_id.astype(str))),
                'heldout_ids':valid.id.astype(str).tolist(),'pred':pred.tolist(),
                'heldout_row_ids':valid.id.astype(str).tolist()}
    dest=OUT/f'{arm}_fold{fold}.joblib'; joblib.dump(checkpoint,dest)
    return checkpoint

def pilot(frame, folds):
    # Build manifest from the ACTUAL fit slices, never from a post-hoc complement constructor.
    frame=frame.copy()
    frame['id']=frame.id.astype(str); frame['prompt_id']=frame.prompt_id.astype(str)
    frame['fold']=frame.prompt_id.map(folds)
    assert frame.fold.notna().all() and frame.id.is_unique
    assert set(frame.label.astype(int))==set(range(18))
    manifests={a:{} for a in ('C0','C1','C2')}
    preds={a:{} for a in manifests}
    for fold in sorted(set(folds.values())):
        train=frame[frame.fold!=fold].copy()
        valid=frame[frame.fold==fold].copy()
        assert not set(train.prompt_id)&set(valid.prompt_id)
        assert set(train.label.astype(int))==set(range(18))
        for arm in manifests:
            ck=fit_one(train,valid,arm,fold)
            manifests[arm][int(fold)]={k:ck[k] for k in ('train_ids','train_prompt_ids','heldout_ids')}
            preds[arm].update(zip(ck['heldout_row_ids'],ck['pred']))
    records=[]
    for row in frame.itertuples(index=False):
        records.append(dict(id=row.id,prompt_id=row.prompt_id,fold=int(row.fold),
                            source='official-dev',label=int(row.label),
                            **{a:int(preds[a][row.id]) for a in preds}))
    gate_mod.validate_fit_provenance(records,folds,manifests)
    # The existing gate's sibling_macro is sibling-CLASS recall, not conditional pair discrimination.
    # A separate true pair-conditioned diagnostic requires knowing the top-two class scores.
    family={i:i//2 for i in range(18)} # PLACEHOLDER ONLY; replace with verified label->family map before interpreting sibling metric
    result=gate_mod.gate(records,folds,family,manifests)
    # Never interpret the placeholder family metric as real model-family discrimination.
    (OUT/'manifest.json').write_text(json.dumps(manifests,indent=2))
    (OUT/'predictions.json').write_text(json.dumps(records,indent=2))
    (OUT/'metrics.json').write_text(json.dumps(result,indent=2))
    return result


In [ ]:
# CPU dry-run: exercises real fit_one train indices, not post-hoc manifests.
# Uses 5 folds x 18 classes x 2 examples, with disjoint synthetic prompts.
def synthetic():
    rows=[]; folds={}
    for f in range(5):
        for p in range(2):
            prompt=f'synth_{f}_{p}'; folds[prompt]=f
            for y in range(18):
                rows.append(dict(id=f'{prompt}_{y}',prompt_id=prompt,label=y,
                    text=f'common topic_{p} author_{y} token_{y} token_{y} sentence end'))
    return pd.DataFrame(rows),folds
synth,synth_folds=synthetic()
res=pilot(synth,synth_folds)
assert len(json.loads((OUT/'predictions.json').read_text()))==180
assert len(json.loads((OUT/'manifest.json').read_text())['C1'])==5
print('DRY_RUN_OK: fit-derived manifests, 3 arms x 5 folds, 180 heldout rows; synthetic metrics NOT performance evidence')


In [ ]:
# REAL DATA: explicit schema and frozen fold contract, no auto-download of private Kaggle data.
# Expected train.csv columns: id, prompt_id, label, text; adapt locally only after inspecting exact official schema.
# Expected frozen_prompt_folds.json: {"prompt_id_string": 0..4, ...}
if DATA.exists() and FOLDS.exists():
    official=pd.read_csv(DATA)
    assert {'id','prompt_id','label','text'}<=set(official.columns), 'Adapt exact official schema, do not guess'
    frozen={str(k):int(v) for k,v in json.loads(FOLDS.read_text()).items()}
    assert len(official)==2699 and official.prompt_id.nunique()==150, 'Dataset does not match historical 2699/150 contract'
    assert set(official.prompt_id.astype(str))==set(frozen), 'Folds not identical to official prompts'
    # Keep synthetic outputs separate, never let a synthetic checkpoint count as a real result.
    OUT=Path('/content/ntl_private/wordchoice_official'); OUT.mkdir(parents=True,exist_ok=True)
    print(pilot(official,frozen))
else:
    print('COLAB_INTERACTIVE_REQUIRED: supply official data and exact frozen fold JSON privately. No real model training occurred.')
